# 3. PraPemrosesan


### **PTA**

**Stopwords**

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re


nltk.download("stopwords")

# Load dataset hasil crawling
df = pd.read_csv("pta_hasil.csv")

# Stopwords bahasa Indonesia
stopwords_indo = set(stopwords.words("indonesian"))

def clean_text(text):
    if pd.isna(text):  # kalau kosong
        return ""
    # lowercase
    text = text.lower()
    # hapus karakter non-huruf
    text = re.sub(r"[^a-zA-Záéíóúàèìòùüñç\s]", " ", text)
    # tokenisasi
    tokens = text.split()
    # hapus stopword
    tokens = [word for word in tokens if word not in stopwords_indo]
    return " ".join(tokens)

# Terapkan hanya ke abstrak indo
df["abstrak_indo_clean"] = df["abstrak_indo"].apply(clean_text)

# Simpan hasil ke file baru
df.to_csv("pta_abstrak_clean.csv", index=False, encoding="utf-8-sig")

print(df[["abstrak_indo", "abstrak_indo_clean"]].head(10))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


                                        abstrak_indo  \
0  ABSTRAK\r\n\r\n       Implementasi Fungsi Legi...   
1  Badan Usaha Milik Negara (BUMN) adalah Badan u...   
2  Kasus narkoba tidak henti-hentinya terdengar d...   
3  \r\nProduk elektronik adalah suatu benda berge...   
4                                                      
5  ABSTRAK\r\n\r\nJaminan akan kemerdekaan dan ke...   
6  ABSTRAK\r\n\r\nPendaftaran tanah untuk pertama...   
7  \r\nABSTRAK\r\n\r\n\tSalah satu bentuk kemajua...   
8  ABSTRAK\r\n\r\nPelaksanaan terhadap kebijakan ...   
9  Oli bekas kapal merupakan oli sisa yang dihasi...   

                                  abstrak_indo_clean  
0  abstrak implementasi fungsi legislasi dprd kab...  
1  badan usaha milik negara bumn badan usaha moda...  
2  narkoba henti hentinya terdengar media televis...  
3  produk elektronik benda bergerak dihasilkan pr...  
4                                                     
5  abstrak jaminan kemerdekaan kebebasan pers ind... 

**Menghilangkan Symbol atau tanda baca**


In [ ]:
import pandas as pd
import re
import string

# Load dataset hasil preprocessing sebelumnya
df = pd.read_csv("pta_abstrak_clean.csv")

def remove_symbols(text):
    if pd.isna(text):
        return ""
    # hapus tanda baca standar (. , ! ? dll)
    text = text.translate(str.maketrans("", "", string.punctuation))
    # hapus karakter non-huruf (biar nggak ada simbol aneh)
    text = re.sub(r"[^a-zA-Záéíóúàèìòùüñç\s]", " ", text)
    # rapikan spasi
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Terapkan ke kolom hasil sebelumnya
df["abstrak_nosymbol"] = df["abstrak_indo_clean"].apply(remove_symbols)

# Simpan hasil baru
df.to_csv("PTAabstrak_nosymbol.csv", index=False, encoding="utf-8-sig")

# Preview hasil
print(df[["abstrak_indo_clean", "abstrak_nosymbol"]].head(10))

                                  abstrak_indo_clean  \
0  abstrak implementasi fungsi legislasi dprd kab...   
1  badan usaha milik negara bumn badan usaha moda...   
2  narkoba henti hentinya terdengar media televis...   
3  produk elektronik benda bergerak dihasilkan pr...   
4                                                NaN   
5  abstrak jaminan kemerdekaan kebebasan pers ind...   
6  abstrak pendaftaran tanah kali bertujuan kepas...   
7  abstrak salah bentuk kemajuan teknologi jual b...   
8  abstrak pelaksanaan kebijakan moratorium diber...   
9  oli bekas kapal oli sisa dihasilkan kegiatan p...   

                                    abstrak_nosymbol  
0  abstrak implementasi fungsi legislasi dprd kab...  
1  badan usaha milik negara bumn badan usaha moda...  
2  narkoba henti hentinya terdengar media televis...  
3  produk elektronik benda bergerak dihasilkan pr...  
4                                                     
5  abstrak jaminan kemerdekaan kebebasan pers ind... 

**Spell Checker**

In [ ]:
!pip install Sastrawi pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 83.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from spellchecker import SpellChecker

# Load dataset hasil sebelumnya
df = pd.read_csv("PTAabstrak_nosymbol.csv")

# Spell checker (default bahasa Inggris, kita isi dengan kosakata dataset)
spell = SpellChecker(language=None)
unique_words = set(" ".join(df["abstrak_nosymbol"].astype(str)).split())
spell.word_frequency.load_words(unique_words)

def spellcheck_text(text):
    if pd.isna(text):
        return ""
    tokens = text.split()
    corrected_tokens = []
    for token in tokens:
        if token not in spell:
            correction = spell.correction(token)
            corrected_tokens.append(correction if correction else token)
        else:
            corrected_tokens.append(token)
    return " ".join(corrected_tokens)

# Terapkan spellchecker ke abstrak
df["abstrak_spellchecked"] = df["abstrak_nosymbol"].apply(spellcheck_text)

# Simpan hasil
df.to_csv("PTAabstrak_spellchecked.csv", index=False, encoding="utf-8-sig")

print(df[["abstrak_nosymbol", "abstrak_spellchecked"]].head(10))

                                    abstrak_nosymbol  \
0  abstrak implementasi fungsi legislasi dprd kab...   
1  badan usaha milik negara bumn badan usaha moda...   
2  narkoba henti hentinya terdengar media televis...   
3  produk elektronik benda bergerak dihasilkan pr...   
4                                                NaN   
5  abstrak jaminan kemerdekaan kebebasan pers ind...   
6  abstrak pendaftaran tanah kali bertujuan kepas...   
7  abstrak salah bentuk kemajuan teknologi jual b...   
8  abstrak pelaksanaan kebijakan moratorium diber...   
9  oli bekas kapal oli sisa dihasilkan kegiatan p...   

                                abstrak_spellchecked  
0  abstrak implementasi fungsi legislasi dprd kab...  
1  badan usaha milik negara bumn badan usaha moda...  
2  narkoba henti hentinya terdengar media televis...  
3  produk elektronik benda bergerak dihasilkan pr...  
4                                                     
5  abstrak jaminan kemerdekaan kebebasan pers ind... 

**Stemming**

In [ ]:
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Load hasil spellchecker
df = pd.read_csv("PTAabstrak_spellchecked.csv")

# Buat stemmer bahasa Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming_text(text):
    if pd.isna(text):
        return ""
    return " ".join([stemmer.stem(word) for word in text.split()])

# Terapkan stemming
df["abstrak_stemmed"] = df["abstrak_spellchecked"].apply(stemming_text)

# Simpan hasil akhir
df.to_csv("PTAabstrak_stemmed.csv", index=False, encoding="utf-8-sig")

print(df[["abstrak_spellchecked", "abstrak_stemmed"]].head(10))

                                abstrak_spellchecked  \
0  abstrak implementasi fungsi legislasi dprd kab...   
1  badan usaha milik negara bumn badan usaha moda...   
2  narkoba henti hentinya terdengar media televis...   
3  produk elektronik benda bergerak dihasilkan pr...   
4                                                NaN   
5  abstrak jaminan kemerdekaan kebebasan pers ind...   
6  abstrak pendaftaran tanah kali bertujuan kepas...   
7  abstrak salah bentuk kemajuan teknologi jual b...   
8  abstrak pelaksanaan kebijakan moratorium diber...   
9  oli bekas kapal oli sisa dihasilkan kegiatan p...   

                                     abstrak_stemmed  
0  abstrak implementasi fungsi legislasi dprd kab...  
1  badan usaha milik negara bumn badan usaha moda...  
2  narkoba henti henti dengar media televisi radi...  
3  produk elektronik benda gerak hasil proses pro...  
4                                                     
5  abstrak jamin merdeka bebas pers indonesia pij... 

**Tokenizing**

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords

# Download resource NLTK (sekali saja)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# Load hasil stemming
df = pd.read_csv("PTAabstrak_stemmed.csv")

# Stopwords bahasa Indonesia
stopwords_indo = set(stopwords.words("indonesian"))

def tokenize_text(text):
    if pd.isna(text):
        return []
    # tokenisasi
    tokens = nltk.word_tokenize(text)
    # buang stopword biar list lebih bersih
    tokens = [t for t in tokens if t.lower() not in stopwords_indo]
    return tokens

# Terapkan tokenisasi → hasil kolom berupa LIST
df["abstrak_tokens"] = df["abstrak_stemmed"].apply(tokenize_text)

# Simpan hasil
df.to_csv("PTAabstrak_tokenized.csv", index=False, encoding="utf-8-sig")

# Preview hasil
print(df["abstrak_tokens"].head(10))
print(type(df["abstrak_tokens"].iloc[0]))  # cek tipe data (harus list)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0    [abstrak, implementasi, fungsi, legislasi, dpr...
1    [badan, usaha, milik, negara, bumn, badan, usa...
2    [narkoba, henti, henti, dengar, media, televis...
3    [produk, elektronik, benda, gerak, hasil, pros...
4                                                   []
5    [abstrak, jamin, merdeka, bebas, pers, indones...
6    [abstrak, daftar, tanah, kali, tuju, hukum, ha...
7    [abstrak, salah, bentuk, maju, teknologi, jual...
8    [abstrak, laksana, bijak, moratorium, laku, pe...
9    [oli, bekas, kapal, oli, sisa, hasil, giat, me...
Name: abstrak_tokens, dtype: object
<class 'list'>


csv ke list

In [ ]:
import pandas as pd
import ast

# Load hasil tokenisasi
df = pd.read_csv("PTAabstrak_tokenized.csv")

# Karena list disimpan sebagai string di CSV, perlu diubah balik ke list
df["abstrak_tokens"] = df["abstrak_tokens"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

# Gabungkan semua token ke dalam satu list besar
all_tokens = [token for tokens in df["abstrak_tokens"] for token in tokens]

print("Jumlah total token:", len(all_tokens))
print("50 token pertama:", all_tokens[:50])

Jumlah total token: 39993
50 token pertama: ['abstrak', 'implementasi', 'fungsi', 'legislasi', 'dprd', 'kabupaten', 'bangkal', 'periode', 'bentuk', 'atur', 'daerah', 'undang', 'undang', 'dasar', 'negara', 'republik', 'indonesia', 'undang', 'undang', 'republik', 'indonesia', 'nomor', 'bentuk', 'atur', 'undang', 'undang', 'undang', 'undang', 'republik', 'indonesia', 'nomor', 'perintah', 'daerah', 'fungsi', 'legislasi', 'tang', 'dprd', 'fungsi', 'legislasi', 'milik', 'dprd', 'fungsi', 'bentuk', 'atur', 'daerah', 'perda', 'begitupula', 'dprd', 'kabupaten', 'bangkal']


**Perhitungan Frekuensi Pada Abstrak PTA**

In [ ]:
import pandas as pd
from collections import Counter
import ast

# Load dataset hasil tokenisasi
df = pd.read_csv("PTAabstrak_tokenized.csv")

# Pastikan kolom list token dibaca benar
df["abstrak_tokens"] = df["abstrak_tokens"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

# Gabungkan semua token jadi satu list
all_tokens = [token for tokens in df["abstrak_tokens"] for token in tokens]

# Hitung frekuensi kata
freq = Counter(all_tokens)

# Ubah ke DataFrame dan urutkan berdasarkan frekuensi
freq_df = pd.DataFrame(freq.items(), columns=["kata", "frekuensi"]).sort_values(by="frekuensi", ascending=False)

# Simpan ke file CSV
output_path = "frekuensi_kata.csv"
freq_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Frekuensi kata berhasil disimpan ke: {output_path}\n")
print(freq_df.head(50).to_string(index=False))

✅ Frekuensi kata berhasil disimpan ke: frekuensi_kata.csv

      kata  frekuensi
    teliti       1021
     hasil        511
      data        438
    metode        310
  pengaruh        260
  analisis        252
      ajar        249
     kerja        236
      tuju        235
     kunci        230
    sistem        229
     usaha        228
     nilai        221
   tingkat        213
       uji        194
     siswa        193
 kabupaten        188
   kembang        183
       the        181
masyarakat        169
     dasar        164
  variabel        153
   bangkal        146
     milik        146
      desa        141
    madura        133
    faktor        133
    teknik        131
    proses        129
 informasi        128
     salah        127
     media        125
 indonesia        121
        of        119
     hukum        114
       air        112
         x        111
      uang        110
     camat        110
         s        109
    undang        107
    daerah       

### **Berita**

**Stopwords**

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re

# Download stopwords Indonesia
nltk.download("stopwords")
stopwords_indo = set(stopwords.words("indonesian"))

# Load dataset CNN
df = pd.read_csv("cnn_berita_rss_full.csv")

def clean_text(text):
    if pd.isna(text):  # kalau kosong
        return ""
    # hapus HTML tags
    text = re.sub(r"<.*?>", " ", text)
    # lowercase
    text = text.lower()
    # hapus karakter non-huruf
    text = re.sub(r"[^a-zA-Záéíóúàèìòùüñç\s]", " ", text)
    # tokenisasi
    tokens = text.split()
    # hapus stopword
    tokens = [word for word in tokens if word not in stopwords_indo]
    return " ".join(tokens)

# Terapkan hanya ke kolom isi_singkat
df["isi_clean"] = df["isi_singkat"].apply(clean_text)

# Simpan hasil
df.to_csv("cnn_clean.csv", index=False, encoding="utf-8-sig")

# Preview hasil
print(df[["isi_singkat", "isi_clean"]].head(10))

                                         isi_singkat  \
0  <img src="https://akcdn.detik.net.id/visual/20...   
1  <img src="https://akcdn.detik.net.id/visual/20...   
2  <img src="https://akcdn.detik.net.id/visual/20...   
3  <img src="https://akcdn.detik.net.id/visual/20...   
4  <img src="https://akcdn.detik.net.id/visual/20...   
5  <img src="https://akcdn.detik.net.id/visual/20...   
6  <img src="https://akcdn.detik.net.id/visual/20...   
7  <img src="https://akcdn.detik.net.id/visual/20...   
8  <img src="https://akcdn.detik.net.id/visual/20...   
9  <img src="https://akcdn.detik.net.id/visual/20...   

                                           isi_clean  
0  kpk menyebut agen travel haji menjual kuota ha...  
1  kpk eks menag yaqut menerbitkan sk kuota haji ...  
2  keberadaan terpidana silfester matutina mister...  
3  banjir melanda bali jembrana denpasar menyebab...  
4  kpk memeriksa dirjen haji hilman latief jam te...  
5  warga china terlibat pencurian rumah kosong ta... 

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**Menghilangkan Symbol atau tanda baca**


In [ ]:
import pandas as pd
import re
import string

# Load dataset hasil preprocessing CNN sebelumnya
df = pd.read_csv("cnn_clean.csv")

def remove_symbols(text):
    if pd.isna(text):
        return ""
    # hapus tanda baca standar (. , ! ? dll)
    text = text.translate(str.maketrans("", "", string.punctuation))
    # hapus karakter non-huruf (supaya simbol aneh juga hilang)
    text = re.sub(r"[^a-zA-Záéíóúàèìòùüñç\s]", " ", text)
    # rapikan spasi ganda
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Terapkan hanya ke kolom isi_clean
df["isi_nosymbol"] = df["isi_clean"].apply(remove_symbols)

# Simpan hasil baru
df.to_csv("cnn_nosymbol.csv", index=False, encoding="utf-8-sig")

# Preview hasil
print(df[["isi_clean", "isi_nosymbol"]].head(10))


                                           isi_clean  \
0  kpk menyebut agen travel haji menjual kuota ha...   
1  kpk eks menag yaqut menerbitkan sk kuota haji ...   
2  keberadaan terpidana silfester matutina mister...   
3  banjir melanda bali jembrana denpasar menyebab...   
4  kpk memeriksa dirjen haji hilman latief jam te...   
5  warga china terlibat pencurian rumah kosong ta...   
6  orang dilaporkan hilang gelombang demontrasi a...   
7  tunjangan rumah anggota dewan sorotan terungka...   
8  kpk mengungkap dugaan korupsi kuota haji kemen...   
9  hadapan menko yuril direktur lokataru delpedro...   

                                        isi_nosymbol  
0  kpk menyebut agen travel haji menjual kuota ha...  
1  kpk eks menag yaqut menerbitkan sk kuota haji ...  
2  keberadaan terpidana silfester matutina mister...  
3  banjir melanda bali jembrana denpasar menyebab...  
4  kpk memeriksa dirjen haji hilman latief jam te...  
5  warga china terlibat pencurian rumah kosong ta... 

**Spell Checker**

In [ ]:
!pip install Sastrawi pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 65.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from spellchecker import SpellChecker

# Load dataset hasil sebelumnya
df = pd.read_csv("cnn_nosymbol.csv")

# Spell checker (default bahasa Inggris, kita isi dengan kosakata dataset CNN)
spell = SpellChecker(language=None)
unique_words = set(" ".join(df["isi_nosymbol"].astype(str)).split())
spell.word_frequency.load_words(unique_words)

def spellcheck_text(text):
    if pd.isna(text):
        return ""
    tokens = text.split()
    corrected_tokens = []
    for token in tokens:
        if token not in spell:
            correction = spell.correction(token)
            corrected_tokens.append(correction if correction else token)
        else:
            corrected_tokens.append(token)
    return " ".join(corrected_tokens)

# Terapkan spellchecker ke isi berita CNN
df["isi_spellchecked"] = df["isi_nosymbol"].apply(spellcheck_text)

# Simpan hasil
df.to_csv("cnn_spellchecked.csv", index=False, encoding="utf-8-sig")

# Preview hasil
print(df[["isi_nosymbol", "isi_spellchecked"]].head(10))

                                        isi_nosymbol  \
0  kpk menyebut agen travel haji menjual kuota ha...   
1  kpk eks menag yaqut menerbitkan sk kuota haji ...   
2  keberadaan terpidana silfester matutina mister...   
3  banjir melanda bali jembrana denpasar menyebab...   
4  kpk memeriksa dirjen haji hilman latief jam te...   
5  warga china terlibat pencurian rumah kosong ta...   
6  orang dilaporkan hilang gelombang demontrasi a...   
7  tunjangan rumah anggota dewan sorotan terungka...   
8  kpk mengungkap dugaan korupsi kuota haji kemen...   
9  hadapan menko yuril direktur lokataru delpedro...   

                                    isi_spellchecked  
0  kpk menyebut agen travel haji menjual kuota ha...  
1  kpk eks menag yaqut menerbitkan sk kuota haji ...  
2  keberadaan terpidana silfester matutina mister...  
3  banjir melanda bali jembrana denpasar menyebab...  
4  kpk memeriksa dirjen haji hilman latief jam te...  
5  warga china terlibat pencurian rumah kosong ta... 

**Stemming**

In [ ]:
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Load hasil spellchecker CNN
df = pd.read_csv("cnn_spellchecked.csv")

# Buat stemmer bahasa Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming_text(text):
    if pd.isna(text):
        return ""
    return " ".join([stemmer.stem(word) for word in text.split()])

# Terapkan stemming ke kolom isi_spellchecked
df["isi_stemmed"] = df["isi_spellchecked"].apply(stemming_text)

# Simpan hasil akhir
df.to_csv("cnn_stemmed.csv", index=False, encoding="utf-8-sig")

# Preview hasil
print(df[["isi_spellchecked", "isi_stemmed"]].head(10))

                                    isi_spellchecked  \
0  kpk menyebut agen travel haji menjual kuota ha...   
1  kpk eks menag yaqut menerbitkan sk kuota haji ...   
2  keberadaan terpidana silfester matutina mister...   
3  banjir melanda bali jembrana denpasar menyebab...   
4  kpk memeriksa dirjen haji hilman latief jam te...   
5  warga china terlibat pencurian rumah kosong ta...   
6  orang dilaporkan hilang gelombang demontrasi a...   
7  tunjangan rumah anggota dewan sorotan terungka...   
8  kpk mengungkap dugaan korupsi kuota haji kemen...   
9  hadapan menko yuril direktur lokataru delpedro...   

                                         isi_stemmed  
0  kpk sebut agen travel haji jual kuota haji khu...  
1  kpk eks menag yaqut terbit sk kuota haji tamba...  
2  ada pidana silfester matutina misteri sorot pu...  
3  banjir landa bal jembrana denpasar sebab orang...  
4  kpk periksa dirjen haji hilman latief jam kait...  
5  warga china libat curi rumah kosong tangerang ... 

**Tokenizing**

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords

# Download resource NLTK (sekali saja, bisa di-comment setelah ada)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# Load hasil stemming CNN
df = pd.read_csv("cnn_stemmed.csv")

# Stopwords bahasa Indonesia
stopwords_indo = set(stopwords.words("indonesian"))

def tokenize_text(text):
    if pd.isna(text):
        return []
    # tokenisasi
    tokens = nltk.word_tokenize(text)
    # buang stopword
    tokens = [t for t in tokens if t.lower() not in stopwords_indo]
    return tokens

# Terapkan tokenisasi → hasil kolom berupa LIST
df["isi_tokens"] = df["isi_stemmed"].apply(tokenize_text)

# Simpan hasil
df.to_csv("cnn_tokenized.csv", index=False, encoding="utf-8-sig")

# Preview hasil
print(df["isi_tokens"].head(10))
print(type(df["isi_tokens"].iloc[0]))  # cek tipe data (harus list)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0    [kpk, agen, travel, haji, jual, kuota, haji, k...
1    [kpk, eks, menag, yaqut, terbit, sk, kuota, ha...
2    [pidana, silfester, matutina, misteri, sorot, ...
3    [banjir, landa, bal, jembrana, denpasar, orang...
4    [kpk, periksa, dirjen, haji, hilman, latief, j...
5    [warga, china, libat, curi, rumah, kosong, tan...
6    [orang, lapor, hilang, gelombang, demontrasi, ...
7    [tunjang, rumah, anggota, dewan, sorot, tunjan...
8    [kpk, duga, korupsi, kuota, haji, menteri, aga...
9    [hadap, menko, yuril, direktur, lokataru, delp...
Name: isi_tokens, dtype: object
<class 'list'>


csv ke list

In [ ]:
import pandas as pd
import ast

# Load hasil tokenisasi
df = pd.read_csv("cnn_tokenized.csv")

# Karena list disimpan sebagai string di CSV, perlu diubah balik ke list
df["isi_tokens"] = df["isi_tokens"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

# Gabungkan semua token ke dalam satu list besar
all_tokens = [token for tokens in df["isi_tokens"] for token in tokens]

print("Jumlah total token:", len(all_tokens))
print("50 token pertama:", all_tokens[:50])

Jumlah total token: 9673
50 token pertama: ['kpk', 'agen', 'travel', 'haji', 'jual', 'kuota', 'haji', 'khusus', 'jemaah', 'harga', 'beda', 'jual', 'rp', 'rp', 'juta', 'kuota', 'kpk', 'eks', 'menag', 'yaqut', 'terbit', 'sk', 'kuota', 'haji', 'ribu', 'kuota', 'asosiasi', 'haji', 'kemenag', 'pidana', 'silfester', 'matutina', 'misteri', 'sorot', 'publik', 'buntut', 'eksekusi', 'hukum', 'putus', 'adil', 'banjir', 'landa', 'bal', 'jembrana', 'denpasar', 'orang', 'tinggal', 'dunia', 'hujan', 'lebat']


**Perhitungan Frekuensi Pada Isi Berita CNN**

In [ ]:
import pandas as pd
from collections import Counter
import ast

# Load dataset hasil tokenisasi
df = pd.read_csv("cnn_tokenized.csv")

# Pastikan kolom list token dibaca benar
df["isi_tokens"] = df["isi_tokens"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

# Gabungkan semua token jadi satu list
all_tokens = [token for tokens in df["isi_tokens"] for token in tokens]

# Hitung frekuensi kata
freq = Counter(all_tokens)

# Ubah ke DataFrame dan urutkan berdasarkan frekuensi
freq_df = pd.DataFrame(freq.items(), columns=["kata", "frekuensi"]).sort_values(by="frekuensi", ascending=False)

# Simpan ke file CSV
output_path = "frekuensi_kata.csv"
freq_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ Frekuensi kata berhasil disimpan ke: {output_path}\n")
print(freq_df.head(50).to_string(index=False))

✅ Frekuensi kata berhasil disimpan ke: frekuensi_kata.csv

       kata  frekuensi
  indonesia        148
    menteri         76
     timnas         76
          u         58
   presiden         48
         rp         46
      piala         45
     selasa         41
     serang         40
  september         40
     israel         37
      dunia         37
      korea         36
       duga         36
      jabat         36
       laga         34
       uang         33
      resmi         33
      mobil         33
    prabowo         33
      orang         32
    jakarta         32
      salah         31
       asia         31
      senin         31
    lebanon         31
        sri         30
     sadewa         30
    purbaya         30
      yudhi         30
kualifikasi         29
       main         29
    mulyani         29
      milik         28
       temu         28
       kait         28
       demo         27
        dpr         27
     negara         27
    selatan         2